# Advanced: Multi-Strategy Correlation Analysis with PAMOLA.CORE

**Goal**: Master all 3 correlation analysis strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Pairwise correlation analysis with specific methods
- **Strategy 2**: Batch correlation for multiple field pairs
- **Strategy 3**: Correlation matrix for comprehensive field relationships
- Compare correlation methods and effectiveness
- Advanced visualization techniques
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of correlation concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/
        └── 02_correlation_analyzer_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.correlation import (
        CorrelationOperation,
        CorrelationMatrixOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 12 columns)
- Sample records (first 5 rows)
- Column statistics (mix of numeric and categorical fields)

**Dataset features:**
- Multiple numeric fields (advertising, sales, satisfaction, age)
- Multiple categorical fields (region, category, channel, season)
- Intentional correlations (advertising → sales, satisfaction → repeat purchases)

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'correlation_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate correlated numeric fields
    advertising_spend = np.random.uniform(1000, 10000, n_records)
    sales_revenue = advertising_spend * np.random.uniform(2, 5, n_records) + np.random.normal(0, 1000, n_records)
    
    customer_satisfaction = np.random.uniform(1, 5, n_records)
    repeat_purchases = (customer_satisfaction * 20) + np.random.normal(0, 10, n_records)
    repeat_purchases = np.clip(repeat_purchases, 0, 100)
    
    # Generate age with correlation to income
    age = np.random.randint(22, 65, n_records)
    income = (age - 22) * 2000 + np.random.uniform(30000, 80000, n_records)
    
    # Categorical fields
    regions = np.random.choice(['North', 'South', 'East', 'West', 'Central'], n_records)
    product_categories = np.random.choice(['Electronics', 'Clothing', 'Books', 'Home', 'Sports'], n_records)
    sales_channels = np.random.choice(['Online', 'In-Store', 'Phone'], n_records, p=[0.5, 0.3, 0.2])
    seasons = np.random.choice(['Spring', 'Summer', 'Fall', 'Winter'], n_records)
    
    # Generate binary field
    premium_member = np.random.choice(['Yes', 'No'], n_records, p=[0.3, 0.7])
    
    df = pd.DataFrame({
        'transaction_id': [f'T{i:04d}' for i in range(1, n_records + 1)],
        'advertising_spend': advertising_spend.round(2),
        'sales_revenue': sales_revenue.round(2),
        'customer_satisfaction': customer_satisfaction.round(2),
        'repeat_purchases_pct': repeat_purchases.round(2),
        'age': age,
        'annual_income': income.round(2),
        'region': regions,
        'product_category': product_categories,
        'sales_channel': sales_channels,
        'season': seasons,
        'premium_member': premium_member
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:25s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    elif pd.api.types.is_numeric_dtype(df[col]):
        print(f"  {col:25s} ({dtype_str:10s}): min={df[col].min():.2f}, max={df[col].max():.2f}")

print(f"\n💡 Dataset includes multiple correlation types for comprehensive testing!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field assignments for each strategy
   - `strategy1_pair`: Two fields for pairwise analysis (default: advertising_spend, sales_revenue)
   - `strategy2_pairs`: Multiple field pairs for batch analysis
   - `strategy3_fields`: Fields for correlation matrix
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for all fields)
- Field types displayed
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All fields must exist in dataset before executing strategies

**This setup will be reused for all 3 strategies**

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_pair": ("advertising_spend", "sales_revenue"),  # Pairwise analysis
    "strategy2_pairs": [  # Batch analysis
        ("customer_satisfaction", "repeat_purchases_pct"),
        ("age", "annual_income"),
        ("region", "product_category")
    ],
    "strategy3_fields": [  # Correlation matrix
        "advertising_spend",
        "sales_revenue",
        "customer_satisfaction",
        "age",
        "annual_income"
    ]
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")

# Validate strategy 1 pair
field1, field2 = FIELD_CONFIG["strategy1_pair"]
for field in [field1, field2]:
    if field not in df.columns:
        raise ValueError(
            f"❌ Field '{field}' for strategy1 not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
print(f"✓ Strategy 1 pair: {field1} x {field2}")
print(f"  {field1}: {df[field1].dtype}")
print(f"  {field2}: {df[field2].dtype}")

# Validate strategy 2 pairs
print(f"\n✓ Strategy 2 pairs ({len(FIELD_CONFIG['strategy2_pairs'])} pairs):")
for f1, f2 in FIELD_CONFIG["strategy2_pairs"]:
    for field in [f1, f2]:
        if field not in df.columns:
            raise ValueError(
                f"❌ Field '{field}' for strategy2 not found!\n"
                f"Available columns: {', '.join(df.columns)}\n"
                f"Please update FIELD_CONFIG"
            )
    print(f"  - {f1} x {f2}")

# Validate strategy 3 fields
print(f"\n✓ Strategy 3 matrix ({len(FIELD_CONFIG['strategy3_fields'])} fields):")
for field in FIELD_CONFIG["strategy3_fields"]:
    if field not in df.columns:
        raise ValueError(
            f"❌ Field '{field}' for strategy3 not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    print(f"  - {field}")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="correlation_adv_001",
    task_type="multi_strategy_correlation",
    description="Multi-strategy correlation analysis",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Pairwise Correlation Analysis

**How to use:**
- Analyzes correlation between two specific fields
- Auto-detects correlation method based on field types
- Run to execute pairwise correlation strategy

**What you'll see:**
- Configuration summary
- Progress: validation → analysis → metrics → viz → save
- Completion time and status
- Correlation coefficient and significance

**Key parameters:**
- `method=None`: Auto-detects (Pearson, Cramer's V, etc.)
- `null_handling='drop'`: Removes rows with nulls
- Generates scatter plot for numeric-numeric pairs

**Note:** Best for detailed analysis of specific field relationships

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: PAIRWISE CORRELATION ANALYSIS")
print("=" * 80 + "\n")

# Get fields from config
field1_s1, field2_s1 = FIELD_CONFIG["strategy1_pair"]

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Pairwise",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = CorrelationOperation(
    # Core parameters
    field1=field1_s1,
    field2=field2_s1,
    method=None,  # Auto-detect
    
    # Null handling
    null_handling='drop',
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_pairwise',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Display key metrics
if result_s1.metrics:
    print(f"\n📊 Analysis Summary:")
    print(f"   Method: {result_s1.metrics.get('correlation_method', 'N/A')}")
    print(f"   Coefficient: {result_s1.metrics.get('correlation_coefficient', 0):.4f}")
    if 'p_value' in result_s1.metrics:
        print(f"   P-value: {result_s1.metrics.get('p_value', 0):.6f}")
        print(f"   Significant: {result_s1.metrics.get('statistically_significant', False)}")

## STRATEGY 2: Batch Correlation Analysis

**How to use:**
- Analyzes multiple field pairs in one batch
- Efficient for exploring multiple relationships
- Run to execute batch correlation strategy

**What you'll see:**
- Configuration summary (3 pairs)
- Progress for each pair
- Completion time and status
- Summary of all correlations

**Key parameters:**
- Processes 3 field pairs sequentially
- Auto-detects method per pair
- Null handling per pair

**Note:** Efficient for exploring multiple relationships simultaneously

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: BATCH CORRELATION ANALYSIS")
print("=" * 80 + "\n")

# Get pairs from config
pairs_s2 = FIELD_CONFIG["strategy2_pairs"]

print(f"Analyzing {len(pairs_s2)} field pairs...\n")
start_time = time.time()

# Storage for results
results_s2 = {}

# Process each pair
for i, (field1, field2) in enumerate(pairs_s2, 1):
    print(f"\n[{i}/{len(pairs_s2)}] {field1} x {field2}")
    print("-" * 60)
    
    # Setup progress tracker for this pair
    tracker_pair = HierarchicalProgressTracker(
        total=6,
        description=f"Pair {i}",
        unit="steps",
        level=0
    )
    
    # Create operation
    operation_pair = CorrelationOperation(
        field1=field1,
        field2=field2,
        method=None,
        null_handling='drop',
        use_cache=False,
        generate_visualization=True,
        save_output=True,
        force_recalculation=False
    )
    
    # Execute
    result_pair = operation_pair.execute(
        data_source=data_source,
        task_dir=task_dir / f'strategy2_batch/pair_{i}_{field1}_{field2}',
        reporter=task_reporter,
        progress_tracker=tracker_pair,
        **kwargs
    )
    
    # Store result
    results_s2[f"{field1}_{field2}"] = result_pair
    
    # Display summary
    if result_pair.metrics:
        method = result_pair.metrics.get('correlation_method', 'N/A')
        coeff = result_pair.metrics.get('correlation_coefficient', 0)
        print(f"   Method: {method}, Coefficient: {coeff:.4f}")

elapsed_s2 = time.time() - start_time
print(f"\n\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")
print(f"   Processed {len(results_s2)} field pairs")

# Summary
print(f"\n📊 Batch Analysis Summary:")
for pair_name, result in results_s2.items():
    if result.metrics:
        coeff = result.metrics.get('correlation_coefficient', 0)
        print(f"   {pair_name:40s}: {coeff:7.4f}")

## STRATEGY 3: Correlation Matrix

**How to use:**
- Creates full correlation matrix for multiple fields
- Analyzes all pairwise relationships
- Run to execute correlation matrix strategy

**What you'll see:**
- Configuration summary (5 fields)
- Progress: preparation → analysis → matrix creation
- Completion time and status
- Significant correlations list

**Key parameters:**
- `fields`: List of 5 numeric fields
- `min_threshold=0.3`: Minimum for significance
- Generates correlation matrix heatmap

**Note:** Best for comprehensive relationship discovery across multiple fields

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: CORRELATION MATRIX")
print("=" * 80 + "\n")

# Get fields from config
fields_s3 = FIELD_CONFIG["strategy3_fields"]

# Setup progress tracker
tracker_s3 = HierarchicalProgressTracker(
    total=4,
    description="Strategy 3: Matrix",
    unit="steps",
    level=0
)

# Create operation
operation_s3 = CorrelationMatrixOperation(
    fields=fields_s3,
    min_threshold=0.3,
    null_handling='drop',
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_matrix',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Display key metrics
if result_s3.metrics:
    print(f"\n📊 Matrix Summary:")
    print(f"   Fields analyzed: {result_s3.metrics.get('fields_analyzed', 0)}")
    print(f"   Significant correlations: {result_s3.metrics.get('significant_correlations', 0)}")
    print(f"   Min threshold: {result_s3.metrics.get('min_threshold', 0)}")

## Step 4: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and coverage

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Coverage comparison (pairs analyzed)
- Artifact counts per strategy

**Note:** Matrix strategy analyzes most relationships but takes longest

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Pairwise): {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Batch):    {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Matrix):   {elapsed_s3:6.2f}s")
print(f"  Total:                 {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print(f"\n📊 Coverage:")
print(f"  Strategy 1: 1 field pair")
print(f"  Strategy 2: {len(results_s2)} field pairs")
n_pairs_s3 = len(fields_s3) * (len(fields_s3) - 1) // 2
print(f"  Strategy 3: {n_pairs_s3} field pairs (matrix)")

print(f"\n📦 Artifacts Generated:")
print(f"  Strategy 1: {len(result_s1.artifacts)} artifacts")
total_s2_artifacts = sum(len(r.artifacts) for r in results_s2.values())
print(f"  Strategy 2: {total_s2_artifacts} artifacts")
print(f"  Strategy 3: {len(result_s3.artifacts)} artifacts")

## Step 5: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: Correlation methods, coefficients, significance
2. **Correlation Results**: Detailed analysis with p-values
3. **Visualizations**: Scatter plots, boxplots, heatmaps

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_pairwise', 'Strategy 1: Pairwise Correlation'),
    ('strategy2_batch', 'Strategy 2: Batch Correlation (all pairs)'),
    ('strategy3_matrix', 'Strategy 3: Correlation Matrix')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # For Strategy 2, we have subdirectories for each pair
    if dir_name == 'strategy2_batch':
        pair_dirs = sorted(list(strategy_dir.glob('pair_*')))
        print(f"\n📁 Found {len(pair_dirs)} pair subdirectories")
        
        for pair_dir in pair_dirs[:2]:  # Show first 2 pairs only
            print(f"\n  📂 {pair_dir.name}")
            print("  " + "-" * 60)
            
            # 1. Metrics from pair directory
            metrics_dir = pair_dir / 'metrics'
            if metrics_dir.exists():
                metrics_files = list(metrics_dir.glob('*.json'))
                
                if metrics_files:
                    # Exclude data_types files
                    filtered = [f for f in metrics_files if not f.name.startswith("data_types")]
                    
                    if filtered:
                        target_files = filtered
                        print(f"    ✓ Found {len(filtered)} metrics file(s)")
                    else:
                        target_files = metrics_files
                        print(f"    ⚠ Fallback to ALL {len(metrics_files)} file(s)")
                    
                    # Pick latest
                    target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
                    latest_metrics_file = target_files[0]
                    
                    print(f"    📄 {latest_metrics_file.name}")
                    
                    try:
                        with open(latest_metrics_file, 'r') as f:
                            data = json.load(f)
                            metrics = data.get('metrics', {})
                        
                        print(f"       Method: {metrics.get('correlation_method', 'N/A')}")
                        print(f"       Coefficient: {metrics.get('correlation_coefficient', 0):.4f}")
                        if 'p_value' in metrics:
                            print(f"       P-value: {metrics.get('p_value', 0):.6f}")
                        
                    except Exception as e:
                        print(f"       ⚠️  Error: {e}")
            
            # 2. Correlation Results from pair directory
            output_dir = pair_dir / 'output'
            if output_dir.exists():
                corr_files = sorted(
                    list(output_dir.glob('*_correlation_*.json')),
                    key=lambda x: x.stat().st_mtime, reverse=True
                )
                if corr_files:
                    print(f"\n    📄 RESULTS: {corr_files[0].name}")
                    try:
                        with open(corr_files[0], 'r') as f:
                            corr = json.load(f)
                        print(f"       Sample size: {corr.get('sample_size', 0)}")
                        print(f"       Interpretation: {corr.get('interpretation', 'N/A')}")
                    except Exception as e:
                        print(f"       ⚠️  Error: {e}")
            
            # 3. Visualizations from pair directory
            viz_dir = pair_dir / 'visualizations'
            if viz_dir.exists():
                viz_files = sorted(
                    list(viz_dir.glob('*.png')),
                    key=lambda x: x.stat().st_mtime, reverse=True
                )
                
                if viz_files:
                    latest_time = viz_files[0].stat().st_mtime
                    latest_batch = [
                        f for f in viz_files 
                        if abs(f.stat().st_mtime - latest_time) < 10
                    ]
                    
                    print(f"\n    📊 VISUALIZATIONS: {len(latest_batch)} files")
                    for viz in latest_batch[:1]:  # Show first 1 only
                        print(f"       📈 {viz.stem.replace('_', ' ').title()}")
                        try:
                            display(Image(filename=str(viz)))
                        except:
                            print(f"          (Display error)")
        
        if len(pair_dirs) > 2:
            print(f"\n  ... and {len(pair_dirs) - 2} more pairs")
        continue  # Skip to next strategy
    
    # For Strategy 1 and 3 - Process at strategy level
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                if dir_name == 'strategy3_matrix':
                    print(f"   Fields analyzed: {metrics.get('fields_analyzed', 0)}")
                    print(f"   Significant correlations: {metrics.get('significant_correlations', 0)}")
                else:
                    print(f"   Method: {metrics.get('correlation_method', 'N/A')}")
                    print(f"   Coefficient: {metrics.get('correlation_coefficient', 0):.4f}")
                    if 'p_value' in metrics:
                        print(f"   P-value: {metrics.get('p_value', 0):.6f}")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Correlation Results (NEWEST)
    output_dir = strategy_dir / 'output'
    if output_dir.exists():
        result_files = sorted(
            list(output_dir.glob('*.json')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        if result_files:
            print(f"\n📄 RESULTS: {result_files[0].name}")
            try:
                with open(result_files[0], 'r') as f:
                    results = json.load(f)
                
                if dir_name == 'strategy3_matrix':
                    if 'significant_correlations' in results:
                        sig_corr = results['significant_correlations']
                        print(f"   Significant pairs: {len(sig_corr)}")
                        for corr in sig_corr[:3]:  # Show top 3
                            f1 = corr['field1']
                            f2 = corr['field2']
                            val = corr['correlation']
                            print(f"   - {f1} x {f2}: {val:.4f}")
                else:
                    print(f"   Sample size: {results.get('sample_size', 0)}")
                    print(f"   Interpretation: {results.get('interpretation', 'N/A')}")
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 3. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:1]:  # Show first 1 only for space
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Cross-Strategy Analysis

**How to use:**
- Run AFTER all strategies complete
- Compare correlation patterns across strategies

**What you'll see:**
- Method distribution (Pearson, Cramer's V, etc.)
- Correlation strength distribution
- Statistical significance summary
- Strongest relationships identified

**Interpretation:**
- Coefficient > 0.7 = Strong correlation
- P-value < 0.05 = Statistically significant
- Method depends on field types

**Note:** Helps identify most important relationships in your data

In [ ]:
print("\n" + "=" * 80)
print("📊 CROSS-STRATEGY ANALYSIS")
print("=" * 80 + "\n")

# Collect all correlations
all_correlations = []

# From Strategy 1
if result_s1.metrics:
    all_correlations.append({
        'strategy': 'Pairwise',
        'pair': f"{field1_s1} x {field2_s1}",
        'method': result_s1.metrics.get('correlation_method', 'N/A'),
        'coefficient': result_s1.metrics.get('correlation_coefficient', 0),
        'p_value': result_s1.metrics.get('p_value'),
        'significant': result_s1.metrics.get('statistically_significant', False)
    })

# From Strategy 2
for pair_name, result in results_s2.items():
    if result.metrics:
        all_correlations.append({
            'strategy': 'Batch',
            'pair': pair_name.replace('_', ' x ', 1),
            'method': result.metrics.get('correlation_method', 'N/A'),
            'coefficient': result.metrics.get('correlation_coefficient', 0),
            'p_value': result.metrics.get('p_value'),
            'significant': result.metrics.get('statistically_significant', False)
        })

# From Strategy 3 - load from file
matrix_file = task_dir / 'strategy3_matrix' / 'output'
if matrix_file.exists():
    matrix_files = sorted(
        list(matrix_file.glob('correlation_matrix_*.json')),
        key=lambda x: x.stat().st_mtime, reverse=True
    )
    if matrix_files:
        try:
            with open(matrix_files[0], 'r') as f:
                matrix_data = json.load(f)
            for corr in matrix_data.get('significant_correlations', []):
                all_correlations.append({
                    'strategy': 'Matrix',
                    'pair': f"{corr['field1']} x {corr['field2']}",
                    'method': 'pearson',  # Matrix uses numeric fields
                    'coefficient': corr['correlation'],
                    'p_value': corr.get('p_value'),
                    'significant': corr.get('statistically_significant', False)
                })
        except Exception as e:
            print(f"Warning: Could not load matrix data: {e}")

if all_correlations:
    # Create DataFrame for analysis
    corr_df = pd.DataFrame(all_correlations)
    
    print("📋 METHOD DISTRIBUTION:")
    print("=" * 80)
    method_counts = corr_df['method'].value_counts()
    for method, count in method_counts.items():
        print(f"  {method:20s}: {count:3d} pairs")
    
    print(f"\n\n📈 CORRELATION STRENGTH DISTRIBUTION:")
    print("=" * 80)
    corr_df['abs_coefficient'] = corr_df['coefficient'].abs()
    
    bins = [0, 0.3, 0.5, 0.7, 1.0]
    labels = ['Weak (<0.3)', 'Moderate (0.3-0.5)', 'Strong (0.5-0.7)', 'Very Strong (>0.7)']
    corr_df['strength'] = pd.cut(corr_df['abs_coefficient'], bins=bins, labels=labels)
    
    strength_counts = corr_df['strength'].value_counts().sort_index()
    for strength, count in strength_counts.items():
        print(f"  {strength:25s}: {count:3d} pairs")
    
    print(f"\n\n🔬 STATISTICAL SIGNIFICANCE:")
    print("=" * 80)
    sig_count = corr_df['significant'].sum()
    total_count = len(corr_df)
    print(f"  Significant (p < 0.05):   {sig_count}/{total_count} ({sig_count/total_count*100:.1f}%)")
    print(f"  Not significant:          {total_count - sig_count}/{total_count}")
    
    print(f"\n\n🏆 STRONGEST CORRELATIONS:")
    print("=" * 80)
    top_corr = corr_df.nlargest(5, 'abs_coefficient')
    for i, row in top_corr.iterrows():
        sig_marker = "✓" if row['significant'] else "✗"
        print(f"  {sig_marker} {row['pair']:40s}: {row['coefficient']:7.4f} ({row['method']})")
else:
    print("⚠️  No correlation data available for analysis")

## Step 7: Export Analysis Results

**How to use:**
- Run AFTER all strategies complete
- Exports analysis results for each strategy

**What you'll see (per strategy):**
- Export path confirmation
- File list (correlations, visualizations)
- File sizes and artifact counts

**Output structure:**
```
advanced_output/
├── strategy1_pairwise/
│   ├── output/          # Correlation JSON
│   └── visualizations/  # Plots PNG
├── strategy2_batch/
│   └── pair_*/ (per pair)
└── strategy3_matrix/
    ├── output/          # Matrix JSON
    └── visualizations/  # Heatmap PNG
```

**Note:** All artifacts preserved for external analysis or reporting

In [ ]:
print("=" * 80)
print("📦 EXPORTING ANALYSIS RESULTS FROM ALL STRATEGIES")
print("=" * 80)

print(f"\n📂 Export base directory: {task_dir}\n")

# Review exported files for each strategy
for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    
    if not strategy_dir.exists():
        print(f"\n⚠️  {title}: Directory not found")
        continue
    
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # For Strategy 2, count subdirectories
    if dir_name == 'strategy2_batch':
        pair_dirs = list(strategy_dir.glob('pair_*'))
        print(f"\n📁 {len(pair_dirs)} pair subdirectories:")
        
        total_files = 0
        for pair_dir in pair_dirs:
            files = list(pair_dir.rglob('*.*'))
            total_files += len(files)
        
        print(f"   Total files: {total_files}")
        print(f"   Average per pair: {total_files/len(pair_dirs):.1f}")
        continue
    
    # Count files in each subdirectory
    for subdir_name in ['output', 'visualizations']:
        subdir = strategy_dir / subdir_name
        if subdir.exists():
            files = list(subdir.glob('*'))
            print(f"\n📁 {subdir_name}/ ({len(files)} files):")
            
            for f in files:
                size_kb = f.stat().st_size / 1024
                print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n\n" + "=" * 80)
print("✅ EXPORT SUMMARY")
print("=" * 80)
print(f"\n📂 All files saved to: {task_dir}")

if 'elapsed_s1' in locals() and 'elapsed_s2' in locals() and 'elapsed_s3' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total processing time: {total_time:.2f}s")

print("\n" + "=" * 80)
print("🎉 EXPORT COMPLETE!")
print("=" * 80)

## 🎯 Summary

**Accomplished:**
- ✅ 3 correlation strategies implemented and compared
- ✅ Pairwise, batch, and matrix analysis completed
- ✅ Multiple correlation methods applied automatically
- ✅ Statistical significance evaluated
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Pairwise analysis provides detailed insights for specific relationships
- Batch analysis efficiently processes multiple field pairs
- Matrix analysis reveals comprehensive relationship patterns
- Method selection depends on field types (numeric, categorical, mixed)
- P-values indicate statistical reliability of correlations

**Strategy recommendations:**
- **Use Pairwise** when: Deep analysis of specific relationships needed
- **Use Batch** when: Multiple specific hypotheses to test
- **Use Matrix** when: Exploring all relationships, discovering patterns

**Correlation methods:**
- **Pearson**: Numeric-numeric (linear relationships)
- **Cramer's V**: Categorical-categorical associations
- **Correlation Ratio**: Categorical-numeric relationships
- **Point-biserial**: Binary-numeric relationships

**Next steps:**
- Apply strategies to your production datasets
- Use matrix analysis for feature selection
- Investigate strong correlations for causality
- Integrate into data quality pipelines

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)